# Structural changes — save visualised results

Explore parameters in [notebook 03](03_structural_changes_visualization.ipynb), then copy the selected values into `PARAMETERS` below and Run All. Widget values are not automatically transferred between kernels.

This notebook reuses notebook 03's plotting helper without running its widgets. Each export creates a new UTC run directory under `outputs/structural_changes/visualizations/` containing PNG/PDF figures, the generated stream CSV and a JSON manifest. These are exploratory visualisations, not detector confirmations or paper reproduction results.


In [1]:
import sys
import json
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

cwd = Path.cwd().resolve()
project_root = next(
    (p for p in (cwd, *cwd.parents)
     if (p / "src/simulation/structural_changes.py").is_file()), None
)
if project_root is None:
    raise FileNotFoundError("Start this kernel inside the repository checkout.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.simulation.structural_changes import generate_structural_stream

source_notebook = project_root / "notebooks/03_structural_changes_visualization.ipynb"
source = json.loads(source_notebook.read_text(encoding="utf-8"))
helper = next(
    (c for c in source["cells"]
     if c["cell_type"] == "code"
     and "def plot_stream(" in "".join(c["source"])), None
)
if helper is None:
    raise RuntimeError("Notebook 03 no longer contains its plotting helper.")
exec(compile("".join(helper["source"]), str(source_notebook) + ":plotting", "exec"))
sns.set(style="whitegrid")


## Choose a reproducible case

Copy the values used in notebook 03. Current generator scenario names describe intended changes; some scenarios also change other parameters. The manifest records this limitation. Re-running this export preserves earlier runs.


In [2]:
PARAMETERS = {
    "seed": 1,
    "scenario": "combined",
    "n_observations": 1000,
    "change_at": 500,
    "variance_ratio": 4.0,
    "correlation_before": 0.0,
    "correlation_after": 0.7,
    "ar_before": 0.0,
    "ar_after": 0.7,
}
DPI = 200
OUTPUT_BASE = project_root / "outputs/structural_changes/visualizations"


In [3]:
def save_visualized_results(parameters, output_base=OUTPUT_BASE, dpi=DPI):
    stream, metadata = generate_structural_stream(**parameters)
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    run_dir = Path(output_base) / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    figure = plot_stream(stream, metadata, show=False)
    try:
        figure.savefig(run_dir / "structural_changes.png", dpi=dpi, bbox_inches="tight")
        figure.savefig(run_dir / "structural_changes.pdf", bbox_inches="tight")
    finally:
        plt.close(figure)
    pd.DataFrame({
        "time": np.arange(len(stream)),
        "feature_1": stream[:, 0],
        "feature_2": stream[:, 1],
    }).to_csv(run_dir / "stream.csv", index=False)

    def git_value(*arguments):
        try:
            return subprocess.check_output(
                ["git", "-C", str(project_root), *arguments],
                text=True, stderr=subprocess.DEVNULL,
            ).strip()
        except (OSError, subprocess.CalledProcessError):
            return None

    source_paths = [
        source_notebook,
        project_root / "src/simulation/structural_changes.py",
        project_root / "notebooks/04_structural_changes_save_results.ipynb",
    ]
    manifest = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "run_id": run_id,
        "parameters": dict(parameters),
        "generator_metadata": metadata,
        "revision": git_value("rev-parse", "HEAD"),
        "git_status_porcelain": git_value("status", "--porcelain"),
        "source_sha256": {
            str(p.relative_to(project_root)): hashlib.sha256(p.read_bytes()).hexdigest()
            for p in source_paths if p.is_file()
        },
        "versions": {
            "numpy": np.__version__, "pandas": pd.__version__,
            "matplotlib": matplotlib.__version__, "seaborn": sns.__version__,
        },
        "figure_dpi": dpi,
        "interpretation": "Exploratory visualisation only; no detector was evaluated.",
        "generator_limitation": (
            "Scenario labels do not guarantee isolated changes. AR coefficients and "
            "innovation covariance may change together; stationary process covariance "
            "is Q/(1-phi**2), with transition effects."
        ),
        "artifacts_sha256": {
            p.name: hashlib.sha256(p.read_bytes()).hexdigest()
            for p in sorted(run_dir.iterdir()) if p.is_file()
        },
    }
    (run_dir / "manifest.json").write_text(
        json.dumps(manifest, indent=2, allow_nan=False) + "\n", encoding="utf-8"
    )
    return run_dir

saved_run = save_visualized_results(PARAMETERS)
print(f"Saved results: {saved_run}")
print("\n".join(p.name for p in sorted(saved_run.iterdir())))


Saved results: C:\Users\b01085983\Documents\GitHub\ewma_covariate_shift_demo\outputs\structural_changes\visualizations\20261013T155218005525Z
manifest.json
stream.csv
structural_changes.pdf
structural_changes.png


## Review or export another case

Open the printed directory to review the figures and manifest. Change `PARAMETERS` and rerun the export cell to create another run. Files are saved locally in your checkout; commit selected outputs if you want them on GitHub.
